### Load data 

In [1]:
# Load CSEC2017 synthetic + KD2017 datasets 
from datasets import load_from_disk
import datasets
datasets.disable_caching()
import sys
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu") 

KAs = {"0": "miscellaneous (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

from utils.load_data import preprocess
# KDs 
KD_dataset = datasets.load_dataset("csv",data_files={"train": "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/train_data.csv"}, split='train')
KD_dataset = KD_dataset.remove_columns('KSAT ID')
KD_dataset = KD_dataset.map(preprocess)
print(KD_dataset)

from utils.load_data import preprocess_csec, preprocess_csec8
# CSEC2017 specific! CHANGED: train_CSEC2017b to train_CSEC2017c to include class 0 
csec_dataset = datasets.load_dataset("csv",data_files={"train": "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/train_CSEC2017c.csv"}, split='train')
csec_dataset = csec_dataset.remove_columns('Statement Description')
csec_dataset = csec_dataset.remove_columns('label')
csec_dataset = csec_dataset.select_columns(['topics','0','1','2','3','4','5','6','7','8'])
csec_dataset = csec_dataset.map(preprocess_csec)
print(csec_dataset)

/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Map:   0%|          | 0/576 [00:00<?, ? examples/s]

Dataset({
    features: ['0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 576
})


Map:   0%|          | 0/2143 [00:00<?, ? examples/s]

Dataset({
    features: ['topics', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2143
})


In [60]:
# clean datasets and convert to pandas 
import pandas as pd 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from utils.load_data import clean_text

feature_type = 'tf-idf' 

pandas_dataCSEC = pd.DataFrame({'topics': csec_dataset['topics'], 'labels': csec_dataset['labels']})
pandas_dataCSEC['cleaned'] = pandas_dataCSEC['topics'].apply(clean_text)

# remove knowledge of (indiscriminative)
def remove_knowledge_of(example):
    example = example[13:]
    return example

pandas_dataKD = pd.DataFrame({'topics': KD_dataset['Statement Description'], 'labels': KD_dataset['labels']})
pandas_dataKD['cleaned'] = pandas_dataKD['topics'].apply(clean_text).apply(remove_knowledge_of)
# merge fine-tuning datasets 
mergeds = pd.concat([pd.DataFrame({'topics': pandas_dataCSEC['cleaned'], 'labels': pandas_dataCSEC['labels']}), 
                     pd.DataFrame({'topics': pandas_dataKD['cleaned'], 'labels': pandas_dataKD['labels']})]).reset_index() # for
# convert labels to int 
mergeds['labels'] = mergeds['labels'].apply(lambda x: [int(i) for i in x])
# compute tf-idf features on merged dataset 
if feature_type == 'bow': 
    vectorizer = CountVectorizer(min_df= 3, stop_words="english")
if feature_type == 'tf-idf': 
    vectorizer = TfidfVectorizer(min_df= 3, stop_words="english", sublinear_tf=True, norm='l2', ngram_range=(1, 2))
features = vectorizer.fit_transform(mergeds['topics']).toarray()
mergeds['tf_idf'] = list(features)

In [61]:
# To do: Train 5 times and average the resulting metrics. 
# To do: 5 different random seeds as well. 
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def train_RF(X_train, y_train, n_estimators = 10): 
    """ Trains a RF on x_train and y_train and returns the trained model
    x_train = a numpy array 
    y_train = a list of integers """
    rf =  RandomForestClassifier(n_estimators=n_estimators)
    return rf.fit(X_train, y_train)
    
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score 
n_repeats = 5 # number of seeds 
n_splits = 10 # number of k splits 
precisions = []
recalls = [] 
f1s = [] 
accs = []
for _ in range(n_repeats): 
    kf = KFold(n_splits=n_splits,  shuffle=True) 
    precision = 0 
    recall = 0 
    f1 = 0 
    acc = 0 
    for i, (train, test) in enumerate(kf.split(mergeds)):  # split into k-folds
        # extract train and test dataset for k-folds cross-validation
        train_ds = mergeds.loc[train] 
        test_ds = mergeds.loc[test] 
        # train the RF on the training dataset 
        model = train_RF(train_ds['tf_idf'].tolist(), train_ds['labels'].tolist())
        # compute metrics on the validation dataset 
        report = classification_report(test_ds['labels'].tolist(), model.predict(test_ds['tf_idf'].tolist())
                                       ,output_dict=True, zero_division=0)
        accuracy = accuracy_score(test_ds['labels'].tolist(), model.predict(test_ds['tf_idf'].tolist()))
        # gather and normalize metrics by the length of the test dataset 
        acc += (accuracy * len(test_ds)) / len(mergeds)
        precision += (report['macro avg']['precision'] * len(test_ds)) / len(mergeds)
        recall += (report['macro avg']['recall'] * len(test_ds)) / len(mergeds)
        f1 += (report['macro avg']['f1-score'] * len(test_ds)) / len(mergeds)
    
    precisions.append(precision) 
    recalls.append(recall) 
    f1s.append(f1)
    accs.append(acc) 
# print the results
print('precision: ',np.mean(precisions))
print('precision std: ',np.std(precisions))
print('recall: ',np.mean(recalls))
print('recall std: ',np.std(recalls))
print('f1: ',np.mean(f1s))
print('f1 std: ',np.std(f1s))
print('accuracy: ',np.mean(accs))
print('accuracy std: ',np.std(accs))

precision:  0.7084035879790109
precision std:  0.0067613150103077125
recall:  0.44157613195481593
recall std:  0.0069934450968690935
f1:  0.5369680480420236
f1 std:  0.006076247552825106
accuracy:  0.3888194189040088
accuracy std:  0.006418406578410714
